In [1]:
import os
import sqlite3
import pandas as pd
from sqlalchemy.testing import uses_deprecated

db_path='skyguard_master.db'
def data_clean():
    if not os.path.exists(db_path):
        print(f"Hata! {db_path} bulunamadı. Lütfen dosya yolunu kontrol edin.")
        return
    conn=None
    try:
        conn=sqlite3.connect(db_path)
        query="SELECT * FROM flights;"
        df=pd.read_sql(query, conn)

        csv_path='skyguard_master.csv'
        df.to_csv(csv_path, index=False, encoding='utf-8')
        print(f"Başarılı!'{csv_path}' dosyasına aktarıldı.")
    except sqlite3.Error as e:
        print(f"SQLite Hatası! {e}. Lütfen veritabanını kontrol ediniz.")
    finally:
        if conn:
            conn.close()
if __name__=="__main__":
    data_clean()

Başarılı!'skyguard_master.csv' dosyasına aktarıldı.


In [1]:
import pandas as pd
dataset=pd.read_csv("skyguard_master.csv")
print(dataset.isnull().sum())
print(dataset.describe())

time                   0
icao24                 0
lat              3943029
lon              3943029
velocity         4262867
heading          4262867
vertrate         4034340
callsign         4639156
onground               0
alert                  0
spi                    0
squawk           2071702
baroaltitude     1283213
geoaltitude      5181925
lastposupdate    3943029
lastcontact            0
dtype: int64
               time           lat           lon      velocity       heading  \
count  1.672384e+07  1.278081e+07  1.278081e+07  1.246097e+07  1.246097e+07   
mean   1.514397e+09  3.741993e+01 -3.179851e+01  1.842374e+02  1.803823e+02   
std    1.382969e+07  1.783297e+01  7.015395e+01  7.708397e+01  1.020873e+02   
min    1.496686e+09 -4.115675e+01 -1.749664e+02  0.000000e+00  0.000000e+00   
25%    1.500887e+09  3.423592e+01 -9.041621e+01  1.347028e+02  9.052371e+01   
50%    1.523889e+09  4.069331e+01 -4.640732e+01  2.118610e+02  1.800000e+02   
75%    1.527545e+09  4.740689e+01 

In [2]:
print(dataset.columns)

Index(['time', 'icao24', 'lat', 'lon', 'velocity', 'heading', 'vertrate',
       'callsign', 'onground', 'alert', 'spi', 'squawk', 'baroaltitude',
       'geoaltitude', 'lastposupdate', 'lastcontact'],
      dtype='object')


In [3]:
import pandas as pd
kullanilacak_kolonlar = [
    'time', 'icao24', 'lat', 'lon', 'velocity',
    'heading', 'vertrate', 'baroaltitude', 'onground'
]
df_clean=dataset[kullanilacak_kolonlar]
df_clean=df_clean[df_clean['onground'] == 0]
df_clean=df_clean.drop(columns=['onground'])
kritik_kolonlar=['velocity', 'lat', 'lon', 'baroaltitude', 'vertrate', 'heading']
df_clean=df_clean.dropna(subset=kritik_kolonlar)
df_clean=df_clean.sort_values(by=['time'])
print(f"Eski Satır Sayısı: 16,723,840")
print(f"Temizlenmiş Yeni Satır Sayısı: {len(df_clean):,}")

Eski Satır Sayısı: 16,723,840
Temizlenmiş Yeni Satır Sayısı: 11,587,880


In [4]:
from sklearn.preprocessing import StandardScaler
df_clean=df_clean.sort_values(by=['icao24', 'time'])
df_clean['delta_velocity']=df_clean.groupby('icao24')['velocity'].diff().fillna(0)
df_clean['delta_heading']=df_clean.groupby('icao24')['heading'].diff().fillna(0)
df_clean['delta_vertrate']=df_clean.groupby('icao24')['vertrate'].diff().fillna(0)
print("Veriler yapay zeka için ölçeklendiriliyor (Scaling)...")
scaler=StandardScaler()
olceklenicek_kolonlar=[
    'velocity', 'heading', 'vertrate', 'baroaltitude',
    'delta_velocity', 'delta_heading', 'delta_vertrate'
]
df_clean[olceklenicek_kolonlar] = scaler.fit_transform(df_clean[olceklenicek_kolonlar])
print(f"Toplam Sütun Sayısı: {len(df_clean.columns)}")

Veriler yapay zeka için ölçeklendiriliyor (Scaling)...
Toplam Sütun Sayısı: 11


In [5]:
from sklearn.ensemble import IsolationForest
features_columns=['velocity', 'heading', 'vertrate', 'baroaltitude',
    'delta_velocity', 'delta_heading', 'delta_vertrate']
iforest=IsolationForest(
    n_estimators=200,
    max_samples=2048,
    contamination=0.03,
    random_state=42
)
X=df_clean[features_columns]
df_clean['anomaly_iforest']=iforest.fit_predict(X)
normal_sayisi = len(df_clean[df_clean['anomaly_iforest'] == 1])
anomali_sayisi = len(df_clean[df_clean['anomaly_iforest'] == -1])
print(f"Normal Uçuş Kaydı (1): {normal_sayisi:,}")
print(f"Tespit Edilen Anomali (-1): {anomali_sayisi:,}")

Normal Uçuş Kaydı (1): 11,045,594
Tespit Edilen Anomali (-1): 542,286


In [6]:
df_anomaliler=df_clean[df_clean['anomaly_iforest']==-1]
ekstrem_anomali=df_anomaliler.sort_values(
    by=['delta_velocity', 'delta_heading', 'velocity'],
    ascending=[False, False, False]
)
print(ekstrem_anomali[['icao24', 'velocity', 'delta_velocity', 'delta_heading', 'baroaltitude']].head(10))

          icao24   velocity  delta_velocity  delta_heading  baroaltitude
6502712   484b00  35.014856      241.386372      -1.965020     -1.375346
1646848   400b26  33.800224      224.810529       1.846229      0.655041
4510649   896116  33.046149      213.357600      -2.511224      0.728340
8674175   424435  30.822518      211.669818     -16.532132     -1.274560
7268530   4249fd  31.717343      208.285825      11.085984      0.055820
9949704   424336  29.117151      207.593105      -9.733492      0.262890
15045359  424352  29.017967      205.043750      -3.158022      0.874938
12816608  4242be  30.875251      204.770503      -4.220994     -0.629527
7249387   424943  30.029886      190.704646      -3.716463      0.728340
12605505  42434d  28.723473      189.430068      -9.177011     -0.303346


In [14]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, precision_recall_curve
import joblib
import gc
import warnings
warnings.filterwarnings('ignore')


def optimize_dtypes(df: pd.DataFrame) -> pd.DataFrame:
    for col in df.select_dtypes('float64').columns:
        df[col]=df[col].astype('float32')
    for col in df.select_dtypes('int64').columns:
        df[col]=pd.to_numeric(df[col], downcast='integer')
    for col in ['icao24', 'callsign']:
        if col in df.columns:
            df[col]=df[col].astype('category')
    return df


def check_resources(df: pd.DataFrame):
    ram_mb=df.memory_usage(deep=True).sum()/1024**2
    print(f"Veri boyutu:{len(df):,} satır")
    print(f"RAM kullanımı:{ram_mb:.0f} MB")
    print(f"Kolon sayısı:{len(df.columns)}")
    if ram_mb > 4000:
        print("RAM yüksek — sampling önerilebilir")


def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    df=df.copy()
    df=optimize_dtypes(df)
    df=df.sort_values(['icao24', 'time']).reset_index(drop=True)
    df=df[df['onground']==False].copy()
    df=df[df['velocity'].between(0, 600)]
    df=df[df['baroaltitude'].between(-100, 15000)]
    df=df[df['vertrate'].between(-50, 50)]
    df=df[df['heading'].between(0, 360)]
    df['squawk']=pd.to_numeric(df['squawk'], errors='coerce').fillna(0).astype('int16')
    gc.collect()
    return df.reset_index(drop=True)


def engineer_features_chunk(chunk: pd.DataFrame) -> pd.DataFrame:
    chunk=chunk.sort_values('time').copy()
    EMERGENCY_SQUAWKS={7500, 7600, 7700}
    chunk['squawk_emergency']=chunk['squawk'].isin(EMERGENCY_SQUAWKS).astype('int8')
    chunk['alert']=chunk['alert'].astype('int8')
    chunk['spi']=chunk['spi'].astype('int8')
    chunk['any_emergency']=(
        (chunk['squawk_emergency']==1) |
        (chunk['alert']==1) |
        (chunk['spi']==1)
    ).astype('int8')
    chunk['dt']=chunk['time'].diff().fillna(1).clip(lower=1)
    chunk['delta_velocity']=chunk['velocity'].diff().fillna(0)
    chunk['delta_heading']=chunk['heading'].diff().fillna(0)
    chunk['delta_vertrate']=chunk['vertrate'].diff().fillna(0)
    chunk['delta_baroalt']=chunk['baroaltitude'].diff().fillna(0)
    chunk['jerk_velocity']=chunk['delta_velocity'].diff().fillna(0)
    chunk['jerk_vertrate']=chunk['delta_vertrate'].diff().fillna(0)
    chunk['jerk_heading']=chunk['delta_heading'].diff().fillna(0)
    chunk['accel_velocity']=chunk['delta_velocity']/chunk['dt']
    chunk['accel_vertrate']=chunk['delta_vertrate']/chunk['dt']
    chunk['rapid_descent']=((chunk['vertrate'] < -10) & (chunk['delta_vertrate'] < -3)).astype('int8')
    chunk['rapid_climb']=((chunk['vertrate'] > 10) & (chunk['delta_vertrate'] > 3)).astype('int8')
    chunk['heading_instability']=chunk['delta_heading'].abs()/(chunk['velocity'].abs()+1e-6)
    chunk['heading_change_rate']=chunk['delta_heading'].abs()/chunk['dt']
    chunk['high_speed_turn']=(
        (chunk['delta_heading'].abs() > 15) & (chunk['velocity'] > 150)
    ).astype('int8')
    chunk['turn_intensity']=chunk['delta_heading'].abs()*chunk['velocity']/1000.0
    WIN=5
    for col in ['velocity', 'vertrate', 'heading', 'baroaltitude']:
        roll=chunk[col].rolling(WIN, min_periods=2)
        chunk[f'{col}_roll_std']=roll.std().fillna(0).astype('float32')
        chunk[f'{col}_roll_mean']=roll.mean().fillna(chunk[col]).astype('float32')
    for col in ['velocity', 'vertrate', 'baroaltitude']:
        m=chunk[col].mean()
        s=chunk[col].std()
        s=s if s > 1e-6 else 1.0
        chunk[f'{col}_flight_zscore']=((chunk[col]-m)/s).astype('float32')
    return chunk


def engineer_features(df: pd.DataFrame, n_jobs: int=1) -> pd.DataFrame:
    from joblib import Parallel, delayed
    groups=[g for _, g in df.groupby('icao24', observed=True)]
    print(f"Toplam uçuş sayısı: {len(groups):,}")
    print(f"Paralel işlem: {n_jobs} core")
    results=Parallel(n_jobs=n_jobs, prefer='threads')(
        delayed(engineer_features_chunk)(g) for g in groups
    )
    df_out=pd.concat(results, ignore_index=True)
    del results, groups
    gc.collect()
    return df_out


FEATURES_V3=[
    'squawk_emergency', 'alert', 'spi', 'any_emergency',
    'velocity', 'heading', 'vertrate', 'baroaltitude',
    'delta_velocity', 'delta_heading', 'delta_vertrate', 'delta_baroalt',
    'jerk_velocity', 'jerk_vertrate', 'jerk_heading',
    'accel_velocity', 'accel_vertrate',
    'rapid_descent', 'rapid_climb', 'heading_instability',
    'heading_change_rate', 'high_speed_turn', 'turn_intensity',
    'velocity_roll_std', 'vertrate_roll_std',
    'heading_roll_std', 'baroaltitude_roll_std',
    'velocity_flight_zscore', 'vertrate_flight_zscore',
    'baroaltitude_flight_zscore',
]


def smart_sample(df: pd.DataFrame, max_train: int=2_000_000) -> pd.DataFrame:
    if len(df)<=max_train:
        return df
    n_flights=df['icao24'].nunique()
    per_flight=max(1, max_train//n_flights)
    sampled=(
        df.groupby('icao24', observed=True)
          .apply(lambda x: x.sample(min(len(x), per_flight), random_state=42))
          .reset_index(drop=True)
    )
    print(f"Örnekleme: {len(df):,} → {len(sampled):,} satır (eğitim için)")
    return sampled


def train_pipeline(df_clean: pd.DataFrame):
    import time

    print("\n[1/5] Feature engineering başlıyor...")
    t0=time.time()
    df=engineer_features(df_clean, n_jobs=2)
    print(f"Tamamlandı: {(time.time()-t0)/60:.1f} dk")

    print("[2/5] Train/holdout split...")
    df_train, df_holdout=train_test_split(df, test_size=0.20, random_state=42)

    print("[3/5] Eğitim örneklemesi (2M cap)...")
    df_train_sampled=smart_sample(df_train, max_train=2_000_000)
    df_train_sampled=df_train_sampled.dropna(subset=FEATURES_V3)

    print("[4/5] Scaler fit...")
    scaler=StandardScaler()
    X_train=scaler.fit_transform(df_train_sampled[FEATURES_V3])

    print("[5/5] Model eğitimi...")
    t1=time.time()
    contamination=0.01
    print(f"contamination: {contamination:.3%}")
    model=IsolationForest(
        n_estimators=200,
        max_samples=2048,
        contamination=contamination,
        random_state=42,
        n_jobs=-1
    )
    model.fit(X_train)
    print(f"      Model eğitimi: {(time.time()-t1)/60:.1f} dk")
    del X_train, df_train_sampled
    gc.collect()
    return model, scaler, df_holdout, df


ANOMALI_TIPLERI={
    'ani_ivmelenme':  {'delta_velocity': +18.0, 'delta_heading':  0.0, 'vertrate':   0.0},
    'sert_manevra':   {'delta_velocity':   0.0, 'delta_heading': -25.0, 'vertrate':   0.0},
    'ani_inis':       {'delta_velocity':   0.0, 'delta_heading':  0.0,  'vertrate': -35.0},
    'kombine_anomali':{'delta_velocity': +12.0, 'delta_heading': +18.0, 'vertrate': -20.0},
}


def inject_anomalies(df_holdout: pd.DataFrame, n_per_type: int=125) -> pd.DataFrame:
    df_normal=df_holdout.sample(n=min(50_000, len(df_holdout)), random_state=42).copy()
    df_normal['label']=1

    parcalar=[]
    for tip, delta in ANOMALI_TIPLERI.items():
        parca=df_holdout.sample(n=min(n_per_type, len(df_holdout)), random_state=42).copy()
        for sutun, kayma in delta.items():
            if sutun in parca.columns:
                std=df_holdout[sutun].std()
                gurultu=np.random.normal(0, 0.15*std, size=len(parca))
                parca[sutun]=parca[sutun]+kayma+gurultu
        parca['accel_velocity']=parca['delta_velocity']/parca['dt'].clip(lower=1)
        parca['accel_vertrate']=parca['delta_vertrate']/parca['dt'].clip(lower=1)
        parca['jerk_velocity']=parca['delta_velocity'].diff().fillna(0)
        parca['jerk_vertrate']=parca['delta_vertrate'].diff().fillna(0)
        parca['rapid_descent']=((parca['vertrate'] < -10) & (parca['delta_vertrate'] < -3)).astype('int8')
        parca['rapid_climb']=((parca['vertrate'] > 10) & (parca['delta_vertrate'] > 3)).astype('int8')
        parca['heading_instability']=parca['delta_heading'].abs()/(parca['velocity'].abs()+1e-6)
        parca['heading_change_rate']=parca['delta_heading'].abs()/parca['dt'].clip(lower=1)
        parca['high_speed_turn']=((parca['delta_heading'].abs() > 15) & (parca['velocity'] > 150)).astype('int8')
        parca['turn_intensity']=parca['delta_heading'].abs()*parca['velocity']/1000.0
        parca['label']=-1
        parca['anomali_tip']=tip
        parcalar.append(parca)

    df_test=pd.concat([df_normal]+parcalar, ignore_index=True)
    return df_test.sample(frac=1, random_state=42).reset_index(drop=True)

In [15]:
def find_optimal_threshold(model, scaler, df_test, y_true, target_recall=0.90):
    scores=model.decision_function(scaler.transform(df_test[FEATURES_V3]))
    neg_scores=-scores
    y_binary=(y_true==-1).astype(int)
    precision, recall, thresholds=precision_recall_curve(y_binary, neg_scores)
    valid=np.where(recall[:-1] >= target_recall)[0]
    best=valid[np.argmax(precision[valid])] if len(valid) > 0 else np.argmax(recall[:-1])
    threshold=thresholds[best]
    print(f"Optimum eşik: {threshold:.4f} | Recall: {recall[best]:.3f} | Precision: {precision[best]:.3f}")
    return threshold


def evaluate(model, scaler, df_test, y_true, threshold=0.0):
    scores=model.decision_function(scaler.transform(df_test[FEATURES_V3]))
    neg_scores=-scores
    y_binary=(y_true==-1).astype(int)
    y_pred=np.where(neg_scores >= threshold, -1, 1)
    print("\n"+"="*55)
    print("  SKYGUARD v3 — DEĞERLENDİRME RAPORU")
    print("="*55)
    print(classification_report(y_true, y_pred,
          target_names=['Anomali (-1)', 'Normal (1)'], digits=3))
    cm=confusion_matrix(y_true, y_pred, labels=[-1, 1])
    TP, FN, FP, TN=cm[0][0], cm[0][1], cm[1][0], cm[1][1]
    print(f"TP: {TP} | FN: {FN} (kaçırılan) | FP: {FP} | TN: {TN}")
    print(f"AUC-ROC: {roc_auc_score(y_binary, neg_scores):.4f}")
    if 'anomali_tip' in df_test.columns:
        print("\nTip bazlı yakalama:")
        for tip in ANOMALI_TIPLERI:
            mask=df_test['anomali_tip']==tip
            if mask.sum()==0:
                continue
            r=(y_pred[mask.values]==-1).sum()/mask.sum()
            print(f"  {tip:<20}: {r:.1%}")
    return scores, y_pred


if __name__=="__main__":
    import time
    t_start=time.time()
    df_raw=pd.read_csv("skyguard_master.csv")

    print("SKYGUARD v3 BAŞLIYOR")
    df_clean=clean_data(df_raw)
    check_resources(df_clean)
    del df_raw; gc.collect()

    model, scaler, df_holdout, df_full=train_pipeline(df_clean)

    df_test=inject_anomalies(df_holdout.dropna(subset=FEATURES_V3))
    y_true=df_test['label']
    threshold=find_optimal_threshold(model, scaler, df_test, y_true, target_recall=0.90)
    evaluate(model, scaler, df_test, y_true, threshold)

    joblib.dump(model, 'models/skyguard_iforest_v3.joblib')
    joblib.dump(scaler, 'models/skyguard_scaler_v3.joblib')
    joblib.dump(threshold, 'models/skyguard_threshold_v3.joblib')

    print(f"\nToplam süre: {(time.time()-t_start)/60:.1f} dakika")
    print("Kaydedildi: skyguard_iforest_v3 | skyguard_scaler_v3 | skyguard_threshold_v3")

=== SKYGUARD v3 BAŞLIYOR ===
Veri boyutu:11,817,934 satır
RAM kullanımı:607 MB
Kolon sayısı:16

[1/5] Feature engineering başlıyor...
Toplam uçuş sayısı: 31,710
Paralel işlem: 2 core
Tamamlandı: 10.4 dk
[2/5] Train/holdout split...
[3/5] Eğitim örneklemesi (2M cap)...
Örnekleme: 9,454,347 → 1,791,959 satır (eğitim için)
[4/5] Scaler fit...
[5/5] Model eğitimi...
contamination: 1.000%
      Model eğitimi: 0.4 dk
Optimum eşik: -0.1759 | Recall: 0.906 | Precision: 0.068

  SKYGUARD v3 — DEĞERLENDİRME RAPORU
              precision    recall  f1-score   support

Anomali (-1)      0.068     0.906     0.127       500
  Normal (1)      0.999     0.876     0.934     50000

    accuracy                          0.876     50500
   macro avg      0.534     0.891     0.530     50500
weighted avg      0.990     0.876     0.926     50500

TP: 453 | FN: 47 (kaçırılan) | FP: 6193 | TN: 43807
AUC-ROC: 0.9345

Tip bazlı yakalama:
  ani_ivmelenme       : 81.6%
  sert_manevra        : 98.4%
  ani_inis    

In [16]:
import joblib
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import numpy as np

model=joblib.load('models/skyguard_iforest_v3.joblib')
scaler=joblib.load('models/skyguard_scaler_v3.joblib')

threshold=find_optimal_threshold(model, scaler, df_test, y_true, target_recall=0.85)
evaluate(model, scaler, df_test, y_true, threshold)

joblib.dump(threshold, 'models/skyguard_threshold_v3.joblib')

Optimum eşik: -0.1729 | Recall: 0.854 | Precision: 0.070

  SKYGUARD v3 — DEĞERLENDİRME RAPORU
              precision    recall  f1-score   support

Anomali (-1)      0.070     0.854     0.129       500
  Normal (1)      0.998     0.886     0.939     50000

    accuracy                          0.886     50500
   macro avg      0.534     0.870     0.534     50500
weighted avg      0.989     0.886     0.931     50500

TP: 427 | FN: 73 (kaçırılan) | FP: 5688 | TN: 44312
AUC-ROC: 0.9345

Tip bazlı yakalama:
  ani_ivmelenme       : 76.0%
  sert_manevra        : 96.8%
  ani_inis            : 68.8%
  kombine_anomali     : 100.0%


['skyguard_threshold_v3.joblib']

In [18]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, precision_recall_curve
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import joblib
import gc
import warnings
warnings.filterwarnings('ignore')

DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Cihaz: {DEVICE}")

LSTM_FEATURES=[
    'velocity', 'vertrate', 'heading', 'baroaltitude',
    'delta_velocity', 'delta_vertrate', 'delta_heading',
]
WINDOW=20
IF_WEIGHT=0.6
LSTM_WEIGHT=0.4


def optimize_dtypes(df: pd.DataFrame) -> pd.DataFrame:
    for col in df.select_dtypes('float64').columns:
        df[col]=df[col].astype('float32')
    for col in df.select_dtypes('int64').columns:
        df[col]=pd.to_numeric(df[col], downcast='integer')
    for col in ['icao24', 'callsign']:
        if col in df.columns:
            df[col]=df[col].astype('category')
    return df


def check_resources(df: pd.DataFrame):
    ram_mb=df.memory_usage(deep=True).sum()/1024**2
    print(f"Veri boyutu:{len(df):,} satır")
    print(f"RAM kullanımı:{ram_mb:.0f} MB")
    print(f"Kolon sayısı:{len(df.columns)}")
    if ram_mb > 4000:
        print("RAM yüksek — sampling önerilebilir")


def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    df=df.copy()
    df=optimize_dtypes(df)
    df=df.sort_values(['icao24', 'time']).reset_index(drop=True)
    df=df[df['onground']==False].copy()
    df=df[df['velocity'].between(0, 600)]
    df=df[df['baroaltitude'].between(-100, 15000)]
    df=df[df['vertrate'].between(-50, 50)]
    df=df[df['heading'].between(0, 360)]
    df['squawk']=pd.to_numeric(df['squawk'], errors='coerce').fillna(0).astype('int16')
    gc.collect()
    return df.reset_index(drop=True)


def engineer_features_chunk(chunk: pd.DataFrame) -> pd.DataFrame:
    chunk=chunk.sort_values('time').copy()
    EMERGENCY_SQUAWKS={7500, 7600, 7700}
    chunk['squawk_emergency']=chunk['squawk'].isin(EMERGENCY_SQUAWKS).astype('int8')
    chunk['alert']=chunk['alert'].astype('int8')
    chunk['spi']=chunk['spi'].astype('int8')
    chunk['any_emergency']=(
        (chunk['squawk_emergency']==1) |
        (chunk['alert']==1) |
        (chunk['spi']==1)
    ).astype('int8')
    chunk['dt']=chunk['time'].diff().fillna(1).clip(lower=1)
    chunk['delta_velocity']=chunk['velocity'].diff().fillna(0)
    chunk['delta_heading']=chunk['heading'].diff().fillna(0)
    chunk['delta_vertrate']=chunk['vertrate'].diff().fillna(0)
    chunk['delta_baroalt']=chunk['baroaltitude'].diff().fillna(0)
    chunk['jerk_velocity']=chunk['delta_velocity'].diff().fillna(0)
    chunk['jerk_vertrate']=chunk['delta_vertrate'].diff().fillna(0)
    chunk['jerk_heading']=chunk['delta_heading'].diff().fillna(0)
    chunk['accel_velocity']=chunk['delta_velocity']/chunk['dt']
    chunk['accel_vertrate']=chunk['delta_vertrate']/chunk['dt']
    chunk['rapid_descent']=((chunk['vertrate'] < -10) & (chunk['delta_vertrate'] < -3)).astype('int8')
    chunk['rapid_climb']=((chunk['vertrate'] > 10) & (chunk['delta_vertrate'] > 3)).astype('int8')
    chunk['heading_instability']=chunk['delta_heading'].abs()/(chunk['velocity'].abs()+1e-6)
    chunk['heading_change_rate']=chunk['delta_heading'].abs()/chunk['dt']
    chunk['high_speed_turn']=(
        (chunk['delta_heading'].abs() > 15) & (chunk['velocity'] > 150)
    ).astype('int8')
    chunk['turn_intensity']=chunk['delta_heading'].abs()*chunk['velocity']/1000.0
    WIN=5
    for col in ['velocity', 'vertrate', 'heading', 'baroaltitude']:
        roll=chunk[col].rolling(WIN, min_periods=2)
        chunk[f'{col}_roll_std']=roll.std().fillna(0).astype('float32')
        chunk[f'{col}_roll_mean']=roll.mean().fillna(chunk[col]).astype('float32')
    for col in ['velocity', 'vertrate', 'baroaltitude']:
        m=chunk[col].mean()
        s=chunk[col].std()
        s=s if s > 1e-6 else 1.0
        chunk[f'{col}_flight_zscore']=((chunk[col]-m)/s).astype('float32')
    return chunk


def engineer_features(df: pd.DataFrame, n_jobs: int=1) -> pd.DataFrame:
    from joblib import Parallel, delayed
    groups=[g for _, g in df.groupby('icao24', observed=True)]
    print(f"Toplam uçuş sayısı: {len(groups):,}")
    print(f"Paralel işlem: {n_jobs} core")
    results=Parallel(n_jobs=n_jobs, prefer='threads')(
        delayed(engineer_features_chunk)(g) for g in groups
    )
    df_out=pd.concat(results, ignore_index=True)
    del results, groups
    gc.collect()
    return df_out


FEATURES_V3=[
    'squawk_emergency', 'alert', 'spi', 'any_emergency',
    'velocity', 'heading', 'vertrate', 'baroaltitude',
    'delta_velocity', 'delta_heading', 'delta_vertrate', 'delta_baroalt',
    'jerk_velocity', 'jerk_vertrate', 'jerk_heading',
    'accel_velocity', 'accel_vertrate',
    'rapid_descent', 'rapid_climb', 'heading_instability',
    'heading_change_rate', 'high_speed_turn', 'turn_intensity',
    'velocity_roll_std', 'vertrate_roll_std',
    'heading_roll_std', 'baroaltitude_roll_std',
    'velocity_flight_zscore', 'vertrate_flight_zscore',
    'baroaltitude_flight_zscore',
]


def smart_sample(df: pd.DataFrame, max_train: int=2_000_000) -> pd.DataFrame:
    if len(df)<=max_train:
        return df
    n_flights=df['icao24'].nunique()
    per_flight=max(1, max_train//n_flights)
    sampled=(
        df.groupby('icao24', observed=True)
          .apply(lambda x: x.sample(min(len(x), per_flight), random_state=42))
          .reset_index(drop=True)
    )
    print(f"Örnekleme: {len(df):,} → {len(sampled):,} satır (eğitim için)")
    return sampled

class LSTMAutoencoder(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int=64, num_layers: int=2):
        super().__init__()
        self.encoder=nn.LSTM(input_dim, hidden_dim, num_layers,
                             batch_first=True, dropout=0.2)
        self.decoder=nn.LSTM(hidden_dim, hidden_dim, num_layers,
                             batch_first=True, dropout=0.2)
        self.output_layer=nn.Linear(hidden_dim, input_dim)

    def forward(self, x):
        _, (h, c)=self.encoder(x)
        dec_in=h[-1].unsqueeze(1).repeat(1, x.size(1), 1)
        dec_out, _=self.decoder(dec_in)
        return self.output_layer(dec_out)


def build_windows(arr: np.ndarray, window: int=WINDOW) -> np.ndarray:
    n=len(arr)-window+1
    if n <= 0:
        return np.empty((0, window, arr.shape[1]), dtype=np.float32)
    idx=np.arange(window)[None, :]+np.arange(n)[:, None]
    return arr[idx].astype(np.float32)


def train_lstm(df_train: pd.DataFrame, lstm_scaler: StandardScaler,
               epochs: int=20, batch_size: int=2048, lr: float=1e-3) -> LSTMAutoencoder:
    print("\n[LSTM] Pencereler oluşturuluyor...")
    arr=lstm_scaler.transform(df_train[LSTM_FEATURES].fillna(0).values.astype(np.float32))
    windows=build_windows(arr)
    if len(windows)==0:
        raise ValueError("Pencere oluşturulamadı — veri çok kısa")

    # RAM tasarrufu için max 500k pencere
    if len(windows) > 500_000:
        idx=np.random.choice(len(windows), 500_000, replace=False)
        windows=windows[idx]
    print(f"[LSTM] Pencere sayısı: {len(windows):,} | Shape: {windows.shape}")

    dataset=TensorDataset(torch.tensor(windows))
    loader=DataLoader(dataset, batch_size=batch_size, shuffle=True,
                      pin_memory=(DEVICE.type=='cuda'), num_workers=0)

    model=LSTMAutoencoder(input_dim=len(LSTM_FEATURES)).to(DEVICE)
    optimizer=torch.optim.Adam(model.parameters(), lr=lr)
    scheduler=torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)
    criterion=nn.MSELoss()

    print(f"[LSTM] Eğitim başlıyor ({epochs} epoch, {DEVICE})...")
    model.train()
    for epoch in range(1, epochs+1):
        total_loss=0.0
        for (batch,) in loader:
            batch=batch.to(DEVICE)
            optimizer.zero_grad()
            recon=model(batch)
            loss=criterion(recon, batch)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss+=loss.item()
        scheduler.step()
        avg=total_loss/len(loader)
        print(f"  Epoch {epoch:02d}/{epochs} | Loss: {avg:.6f}")

    del windows, dataset, loader
    gc.collect()
    return model


def lstm_reconstruction_errors(model: LSTMAutoencoder, arr: np.ndarray,
                                batch_size: int=2048) -> np.ndarray:
    model.eval()
    windows=build_windows(arr)
    if len(windows)==0:
        return np.zeros(len(arr), dtype=np.float32)

    errors=[]
    with torch.no_grad():
        for i in range(0, len(windows), batch_size):
            batch=torch.tensor(windows[i:i+batch_size]).to(DEVICE)
            recon=model(batch)
            err=((recon-batch)**2).mean(dim=(1, 2)).cpu().numpy()
            errors.append(err)
    errors=np.concatenate(errors)

    # pencere → satır hizalaması: her satır, içinde bulunduğu pencerelerin ortalaması
    full=np.zeros(len(arr), dtype=np.float32)
    count=np.zeros(len(arr), dtype=np.float32)
    for i, e in enumerate(errors):
        full[i:i+WINDOW]+=e
        count[i:i+WINDOW]+=1
    count=np.where(count==0, 1, count)
    return full/count



def train_pipeline(df_clean: pd.DataFrame):
    import time

    print("\n[1/5] Feature engineering başlıyor...")
    t0=time.time()
    df=engineer_features(df_clean, n_jobs=2)
    print(f"Tamamlandı: {(time.time()-t0)/60:.1f} dk")

    print("[2/5] Train/holdout split...")
    df_train, df_holdout=train_test_split(df, test_size=0.20, random_state=42)

    print("[3/5] Eğitim örneklemesi (2M cap)...")
    df_train_sampled=smart_sample(df_train, max_train=2_000_000)
    df_train_sampled=df_train_sampled.dropna(subset=FEATURES_V3)

    print("[4/5] Scaler fit...")
    scaler=StandardScaler()
    X_train=scaler.fit_transform(df_train_sampled[FEATURES_V3])

    print("[5/5] Model eğitimi...")
    t1=time.time()
    contamination=0.01
    print(f"contamination: {contamination:.3%}")
    model=IsolationForest(
        n_estimators=200,
        max_samples=2048,
        contamination=contamination,
        random_state=42,
        n_jobs=-1
    )
    model.fit(X_train)
    print(f"      Model eğitimi: {(time.time()-t1)/60:.1f} dk")
    del X_train, df_train_sampled
    gc.collect()
    return model, scaler, df_train, df_holdout, df


ANOMALI_TIPLERI={
    'ani_ivmelenme':  {'delta_velocity': +18.0, 'delta_heading':  0.0, 'vertrate':   0.0},
    'sert_manevra':   {'delta_velocity':   0.0, 'delta_heading': -25.0, 'vertrate':   0.0},
    'ani_inis':       {'delta_velocity':   0.0, 'delta_heading':  0.0,  'vertrate': -35.0},
    'kombine_anomali':{'delta_velocity': +12.0, 'delta_heading': +18.0, 'vertrate': -20.0},
}


def inject_anomalies(df_holdout: pd.DataFrame, n_per_type: int=125) -> pd.DataFrame:
    df_normal=df_holdout.sample(n=min(50_000, len(df_holdout)), random_state=42).copy()
    df_normal['label']=1

    parcalar=[]
    for tip, delta in ANOMALI_TIPLERI.items():
        parca=df_holdout.sample(n=min(n_per_type, len(df_holdout)), random_state=42).copy()
        for sutun, kayma in delta.items():
            if sutun in parca.columns:
                std=df_holdout[sutun].std()
                gurultu=np.random.normal(0, 0.15*std, size=len(parca))
                parca[sutun]=parca[sutun]+kayma+gurultu
        parca['accel_velocity']=parca['delta_velocity']/parca['dt'].clip(lower=1)
        parca['accel_vertrate']=parca['delta_vertrate']/parca['dt'].clip(lower=1)
        parca['jerk_velocity']=parca['delta_velocity'].diff().fillna(0)
        parca['jerk_vertrate']=parca['delta_vertrate'].diff().fillna(0)
        parca['rapid_descent']=((parca['vertrate'] < -10) & (parca['delta_vertrate'] < -3)).astype('int8')
        parca['rapid_climb']=((parca['vertrate'] > 10) & (parca['delta_vertrate'] > 3)).astype('int8')
        parca['heading_instability']=parca['delta_heading'].abs()/(parca['velocity'].abs()+1e-6)
        parca['heading_change_rate']=parca['delta_heading'].abs()/parca['dt'].clip(lower=1)
        parca['high_speed_turn']=((parca['delta_heading'].abs() > 15) & (parca['velocity'] > 150)).astype('int8')
        parca['turn_intensity']=parca['delta_heading'].abs()*parca['velocity']/1000.0
        parca['label']=-1
        parca['anomali_tip']=tip
        parcalar.append(parca)

    df_test=pd.concat([df_normal]+parcalar, ignore_index=True)
    return df_test.sample(frac=1, random_state=42).reset_index(drop=True)



def find_optimal_threshold(scores: np.ndarray, y_true: pd.Series,
                           target_recall: float=0.90) -> float:
    y_binary=(y_true==-1).astype(int)
    precision, recall, thresholds=precision_recall_curve(y_binary, scores)
    valid=np.where(recall[:-1] >= target_recall)[0]
    best=valid[np.argmax(precision[valid])] if len(valid) > 0 else np.argmax(recall[:-1])
    threshold=thresholds[best]
    print(f"Optimum eşik: {threshold:.4f} | Recall: {recall[best]:.3f} | Precision: {precision[best]:.3f}")
    return threshold


def evaluate(hybrid_scores: np.ndarray, y_true: pd.Series,
             threshold: float, df_test: pd.DataFrame):
    y_binary=(y_true==-1).astype(int)
    y_pred=np.where(hybrid_scores >= threshold, -1, 1)
    print("\n"+"="*55)
    print("  SKYGUARD v3 — DEĞERLENDİRME RAPORU (HYBRID)")
    print("="*55)
    print(classification_report(y_true, y_pred,
          target_names=['Anomali (-1)', 'Normal (1)'], digits=3))
    cm=confusion_matrix(y_true, y_pred, labels=[-1, 1])
    TP, FN, FP, TN=cm[0][0], cm[0][1], cm[1][0], cm[1][1]
    print(f"TP: {TP} | FN: {FN} (kaçırılan) | FP: {FP} | TN: {TN}")
    print(f"AUC-ROC: {roc_auc_score(y_binary, hybrid_scores):.4f}")
    if 'anomali_tip' in df_test.columns:
        print("\nTip bazlı yakalama:")
        for tip in ANOMALI_TIPLERI:
            mask=df_test['anomali_tip']==tip
            if mask.sum()==0:
                continue
            r=(y_pred[mask.values]==-1).sum()/mask.sum()
            print(f"  {tip:<20}: {r:.1%}")
    return y_pred



if __name__=="__main__":
    import time
    t_start=time.time()

    df_raw=pd.read_csv("skyguard_master.csv")
    print("=== SKYGUARD v3 BAŞLIYOR ===")
    df_clean=clean_data(df_raw)
    check_resources(df_clean)
    del df_raw; gc.collect()

    # IF eğitimi
    if_model, if_scaler, df_train, df_holdout, df_full=train_pipeline(df_clean)

    # LSTM eğitimi — sadece normal eğitim verisi üzerinde
    print("\n=== LSTM EĞİTİMİ ===")
    lstm_scaler=StandardScaler()
    df_train_clean=df_train.dropna(subset=LSTM_FEATURES)
    lstm_scaler.fit(df_train_clean[LSTM_FEATURES].values.astype(np.float32))
    lstm_model=train_lstm(df_train_clean, lstm_scaler, epochs=20, batch_size=2048)
    del df_train_clean; gc.collect()

    # Test seti oluştur
    df_test=inject_anomalies(df_holdout.dropna(subset=FEATURES_V3))
    y_true=df_test['label']

    # IF skorları
    if_scores=-if_model.decision_function(
        if_scaler.transform(df_test[FEATURES_V3])
    )

    # LSTM reconstruction error skorları
    print("\n[LSTM] Test seti reconstruction error hesaplanıyor...")
    lstm_arr=lstm_scaler.transform(df_test[LSTM_FEATURES].fillna(0).values.astype(np.float32))
    lstm_errors=lstm_reconstruction_errors(lstm_model, lstm_arr)

    # Normalize et [0,1]
    def minmax(arr):
        mn, mx=arr.min(), arr.max()
        return (arr-mn)/(mx-mn+1e-8)

    if_norm=minmax(if_scores)
    lstm_norm=minmax(lstm_errors)

    hybrid_scores=IF_WEIGHT*if_norm+LSTM_WEIGHT*lstm_norm

    threshold=find_optimal_threshold(hybrid_scores, y_true, target_recall=0.90)
    evaluate(hybrid_scores, y_true, threshold, df_test)
    joblib.dump(if_model, 'models/skyguard_iforest_v3.joblib')
    joblib.dump(if_scaler, 'models/skyguard_scaler_v3.joblib')
    joblib.dump(lstm_scaler, 'models/skyguard_lstm_scaler_v3.joblib')
    joblib.dump(threshold, 'models/skyguard_threshold_v3.joblib')
    torch.save(lstm_model.state_dict(), 'models/skyguard_lstm_v3.pt')

    print(f"\nToplam süre: {(time.time()-t_start)/60:.1f} dakika")
    print("Kaydedildi: skyguard_iforest_v3 | skyguard_scaler_v3 | skyguard_lstm_v3 | skyguard_lstm_scaler_v3 | skyguard_threshold_v3")

Cihaz: cuda
=== SKYGUARD v3 BAŞLIYOR ===
Veri boyutu:11,817,934 satır
RAM kullanımı:607 MB
Kolon sayısı:16

[1/5] Feature engineering başlıyor...
Toplam uçuş sayısı: 31,710
Paralel işlem: 2 core
Tamamlandı: 9.0 dk
[2/5] Train/holdout split...
[3/5] Eğitim örneklemesi (2M cap)...
Örnekleme: 9,454,347 → 1,791,959 satır (eğitim için)
[4/5] Scaler fit...
[5/5] Model eğitimi...
contamination: 1.000%
      Model eğitimi: 0.3 dk

=== LSTM EĞİTİMİ ===

[LSTM] Pencereler oluşturuluyor...
[LSTM] Pencere sayısı: 500,000 | Shape: (500000, 20, 7)
[LSTM] Eğitim başlıyor (20 epoch, cuda)...
  Epoch 01/20 | Loss: 0.959672
  Epoch 02/20 | Loss: 0.816753
  Epoch 03/20 | Loss: 0.710217
  Epoch 04/20 | Loss: 0.667413
  Epoch 05/20 | Loss: 0.645216
  Epoch 06/20 | Loss: 0.629463
  Epoch 07/20 | Loss: 0.622026
  Epoch 08/20 | Loss: 0.616412
  Epoch 09/20 | Loss: 0.610080
  Epoch 10/20 | Loss: 0.603722
  Epoch 11/20 | Loss: 0.597430
  Epoch 12/20 | Loss: 0.593539
  Epoch 13/20 | Loss: 0.588965
  Epoch 14/20 